# Ottawa Neighbourhood Safety — Data Cleaning Pipeline

This notebook prepares a single analysis-ready dataset by ingesting, cleaning, and merging eight Ottawa Police Service open-data files. The output is `CleanedData.csv`: one row per neighbourhood, 24 features, covering 104 neighbourhoods.

---

## Data Sources

| Dataset | Year range | Records |
|---------|------------|---------|
| Auto Theft | 2018–2025 | 11,720 |
| Bike Theft | 2018–2025 | 15,676 |
| Criminal Offences | 2018–2025 | 344,290 |
| Homicide | 2018–2025 | 131 |
| Shootings | 2018–2025 | 505 |
| Hate Crimes | 2018–2024 | 1,689 |
| Calls for Service | 2021–2025 | 1,108,780 |
| Neighbourhood Boundaries (ONS) | — | 111 |

> Hate crime data covers 2018–2024 only — the most recent OPS release at time of analysis.

---

## Pipeline Overview

### Step 1 — Aggregate each dataset by neighbourhood
---

### Step 2 — Merge all crime datasets
---

### Step 3 — Align neighbourhood names and attach police / population data
---

### Step 4 — Drop Greenbelt neighbourhoods and export

**Output: `CleanedData.csv` — 104 neighbourhoods × 24 columns**

| Column group | Columns |
|-------------|---------|
| Identity | `Neighbourhood` |
| Crime counts (18) | `Arson`, `Assaults`, `Attempting The Commission Of A Capital Crime`, `Break and Enter`, `Mischief`, `Offensive Weapons`, `Other Violations Involving Violence Or The Threat Of Violence`, `Possession / Trafficking Stolen Goods`, `Sexual Violations`, `Theft $5000 and Under`, `Theft Over $5000`, `Violations Causing Death`, `Violations Resulting In The Deprivation Of Freedom`, `Auto Theft`, `Bike Theft`, `Homicide`, `Shootings`, `Other` |
| Police response | `Police_Critical`, `Police_High`, `Police_Medium` |
| Demographics | `Population` |
| Geography | `geometry` |

In [ ]:
# import libraries
import pandas as pd
pd.set_option('display.max_columns', None)
import geopandas as gpd

from rapidfuzz import process, fuzz

#### Import the data

In [3]:
files = {
    'auto':'Auto_Theft_Open_Data_-253522024537541393.geojson',
    'bike':'Bike_Theft_Open_Data_5872599271284607536.geojson',
    'calls': 'Calls_For_Service_-4757262503425704098.geojson',
    'crimes': 'Criminal_Offences_Open_Data_-8357852459192701578.geojson',
    'homicide': 'Homicide_Open_Data_5631904391428638505.geojson',
    'neighbourhoods': 'Ottawa_Neighbourhood_Study_(ONS)_-_Neighbourhood_Boundaries_Gen_2.geojson',
    'shooting': 'Shootings_Open_Data_1076553211161030801.geojson',
    'hate': 'Hate_Crime_Open_Data_3325562468580911378.geojson'
}

dfs_geo = {}
for name, path in files.items():
    dfs_geo[name] = gpd.read_file(f'./Dataset/GEO/{path}')
    print(f"{name}: {dfs_geo[name].shape}")

df_auto, df_bike, df_calls, df_crimes, df_homicide, df_neighbourhoods, df_shooting, df_hate = dfs_geo.values()
print('Dataset loaded and stored in variables.')


auto: (11720, 23)
bike: (15676, 27)
calls: (1108780, 10)
crimes: (344290, 18)
homicide: (131, 12)
neighbourhoods: (111, 9)
shooting: (505, 18)
hate: (1689, 18)
Dataset loaded and stored in variables.


#### Inspect the data

Check the year ranges for consistency

In [4]:
year_cols = {
    'auto':     'YEAR',
    'bike':     'YEAR',
    'crimes':   'YEAR',
    'homicide': 'YEAR',
    'hate': 'YEAR',
    'shooting': 'OCC_YEAR',
    'calls':    'RCVD_YEAR',
}

for name, col in year_cols.items():
    years = dfs_geo[name][col]
    print(f"{name}: {years.min()} - {years.max()}")

auto: 2018 - 2025
bike: 2018 - 2025
crimes: 2018 - 2025
homicide: 2018 - 2025
hate: 2018 - 2024
shooting: 2018 - 2025
calls: 2021 - 2025


From the data range we have, we can filter and perform analysis and model trainging for 2018-2025.
Additionally Ottawa Police Service have not released a new hate crime dataset so we will use the 2018-24 data.

### Clean and aggregate datasets by neighbourhood
Each step will perform a count of each record grouped by the neighbourhood.

#### Auto Theft
**Auto Theft → `agg_auto`**: count per neighbourhood; out-of-jurisdiction records dropped.

In [414]:
df_auto.head(1)

,OBJECTID,VEH_YEAR,VEH_MAKE,VEH_MODEL,VEH_STYLE,VEH_COLOUR,VEH_VALUE,WEEKDAY,RECOVERED,NB_NAME_EN,WARD,SECTOR,REP_DATE,OCC_DATE,YEAR,INTERSECTION,DIVISION,CENSUS_TRC,TOD,COUNCILLOR,REP_HOUR,OCC_HOUR,geometry
0,1,2000.0,"HONDA/AMERICAN HONDA MOTOR CO.,",CIVIC (AND CRX),Automobile,DGR,NaN,Monday,Y,Carson Grove - Carson Meadows,Ward 11 - Beacon Hill-Cyrville,Sector 32,"Mon, 01 Jan 2018 05:00:00 GMT","Mon, 01 Jan 2018 05:00:00 GMT",2018,"CARVER PL, CARVER PL",East,5050122.03,Night,Tim Tierney,700,0,POINT Z (-75.62858 45.43323 0)


In [415]:
# aggreate the number of theft by neighbourhood
# handle out of jurisdiction - drop
agg_auto = df_auto[df_auto['NB_NAME_EN'] != '<Data Quality - Out of Jurisdiction>'] \
    .groupby('NB_NAME_EN').agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
agg_auto.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Auto Theft'}, inplace=True)

print(f'We had {agg_auto["Auto Theft"].sum()} auto thefts across {agg_auto.shape[0]} neighbourhoods.')
agg_auto.sort_values('Auto Theft', ascending=False).head()

We had 11189 auto thefts across 111 neighbourhoods.


,Neighbourhood,Auto Theft
72,New Barrhaven - New Development - Stonebridge,605
74,Orléans Avalon - Notting Gate - Fallingbrook -...,587
22,Centretown,413
93,Riverside South - Leitrim,366
33,East Industrial,313


#### Bike Theft
**Bike Theft → `agg_bike`**: count per neighbourhood.

In [416]:
df_bike.head(1)

,OBJECTID,ID,YEAR,REP_DATE,OCC_DATE,DOW,OFF_CATEG,BIKE_STYLE,BIKE_VALUE,BIKE_MAKE,BIKE_MODEL,BIKE_TYPE,BIKE_FRAME,BIKECOLOUR,BIKE_SPEED,NB_NAME_EN,SECTOR,DIVISION,CENSUS_TRC,STATUS,INTERSECTION,TOD,WARD,COUNCILLOR,REP_HOUR,OCC_HOUR,geometry
0,1,1,2018,"Fri, 05 Jan 2018 05:00:00 GMT","Thu, 04 Jan 2018 05:00:00 GMT",Friday,Theft< Bicycle,Men's,800.0,DIVINCI,SILVERSTONE,Racer,None,RED,1,Centretown,Sector 23,Central,5050037.02,Stolen,"MACLAREN ST, METCALFE ST",Evening,Ward 14 - Somerset,Ariel Troster,1100,1900,POINT Z (-75.69177 45.41637 0)


In [417]:
# aggreate the number of bike theft by neighbourhood
agg_bike = df_bike.groupby('NB_NAME_EN').agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
agg_bike.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Bike Theft'}, inplace=True)

print(f'We had {agg_bike["Bike Theft"].sum()} bike thefts across {agg_bike.shape[0]} neighbourhoods.')
agg_bike.sort_values('Bike Theft', ascending=False).head()

We had 15676 bike thefts across 107 neighbourhoods.


,Neighbourhood,Bike Theft
22,Centretown,2437
33,East Industrial,1940
94,Sandy Hill - Ottawa East,1132
13,Byward Market,798
37,Glebe - Dows Lake,728


#### Criminal Offences
**Criminal Offences → `agg_crime`**: count per neighbourhood × offence category, pivoted wide (18 offence types). `Theft - Motor Vehicle` dropped — it duplicates the dedicated auto theft dataset. `Carp` and `Carp Ridge` consolidated into one row — same geographic area, inconsistent naming across datasets.

In this data set we have various offence categories, so we will aggregate by them

In [6]:
print(f'Number of criminal categories: {df_crimes['OFF_CATEG'].nunique()}')
df_crimes.head(1)

Number of criminal categories: 18


,OBJECTID,YEAR,REP_DATE,REP_HOUR,OCC_DATE,OCC_HOUR,WEEKDAY,OFF_SUM,OFF_CATEG,NB_NAME_EN,SECTOR,DIVISION,CENSUS_TRC,TOD,WARD,COUNCILLOR,INTERSECTION,geometry
0,1,2018,"Wed, 24 Jan 2018 05:00:00 GMT",1400,"Wed, 24 Jan 2018 05:00:00 GMT",1400,Wednesday,Crimes Against Property (2000),Theft $5000 and Under,Ledbury - Heron Gate - Ridgemont - Elmwood,Sector 34,East,5050007.02,Afternoon,Ward 18 - Alta Vista,Marty Carr,"WALKLEY RD, HEATHERINGTON RD",POINT Z (-75.64495 45.3779 0)


In [481]:
# aggreate the number of crime by neighbourhood and offence category
total_crime = df_crimes.groupby(['NB_NAME_EN', 'OFF_CATEG']) \
    .agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Count'}) \
    .reset_index()

# rename the columns
total_crime.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Count'}, inplace=True)

# pivot the data to have offence categories as columns and fill any missing values with 0
agg_crime = total_crime.pivot(
    index='Neighbourhood',
    columns='OFF_CATEG',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

print(f'We had {total_crime["Count"].sum()} crimes across {agg_crime.shape[0]} neighbourhoods.')

agg_crime.head()

We had 344104 crimes across 114 neighbourhoods.


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft - Motor Vehicle,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom
0,Barrhaven,3,208,0,79,0,463,258,10,87,171,0,0,59,346,114,9,3,4
1,Bayshore,3,278,1,129,0,500,244,6,152,203,11,0,73,1722,117,42,0,5
2,Beacon Hill South - Cardinal Heights,5,266,1,107,0,409,210,9,137,187,4,0,76,598,83,23,0,9
3,Beaverbrook,1,87,1,34,0,118,97,0,53,85,0,0,27,113,22,10,0,1
4,Beechwood Cemetery,1,1,0,1,0,3,3,0,2,0,1,0,0,4,0,0,0,0


Notice we have a 'Theft-Motor Vehicle' feature which is similar to the auto theft feature in our initial dataset, so we can drop this.

In [482]:
agg_crime.drop(columns = ['Theft - Motor Vehicle'], inplace=True)

During cluster visualization, I noticed this were the same neighbourhood, but had seperate names, so we will considate into one - Carp

In [483]:
agg_crime[agg_crime['Neighbourhood'].isin(['Carp','Carp Ridge'])]

,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom
20,Carp,4,54,0,80,0,192,80,1,22,64,3,0,14,154,24,0,4
21,Carp Ridge,0,3,0,6,0,5,4,1,10,2,0,0,4,5,2,0,0


In [484]:
# consolidate Carp Ridge into Carp in agg_crime
parts = ['Carp', 'Carp Ridge']
subset = agg_crime[agg_crime['Neighbourhood'].isin(parts)]

numeric_cols = [c for c in agg_crime.columns if c != 'Neighbourhood']
combined = subset[numeric_cols].sum().to_dict()
combined['Neighbourhood'] = 'Carp'

agg_crime = agg_crime[~agg_crime['Neighbourhood'].isin(parts)]
agg_crime = pd.concat([agg_crime, pd.DataFrame([combined])], ignore_index=True)

print(f'agg_crime shape: {agg_crime.shape}')
agg_crime[agg_crime['Neighbourhood'] == 'Carp']

agg_crime.head(2)

agg_crime shape: (113, 18)


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom
0,Barrhaven,3,208,0,79,0,463,258,10,87,171,0,0,59,346,9,3,4
1,Bayshore,3,278,1,129,0,500,244,6,152,203,11,0,73,1722,42,0,5


#### Hate Crime
**Hate Crimes → `agg_hate`**: count per neighbourhood × motivation type, pivoted wide (11 categories). Four ambiguous/indeterminate categories (`Combination`, `Not Applicable`, `Other Similar Factor`, `Unknown`) collapsed into a single `Other Hate Crimes` column.

In [7]:
print(f'Number of hate categories: {df_hate['HATE_CRIME'].nunique()}')

df_hate.head(1)

Number of hate categories: 11


,OBJECTID,ID,YEAR,REP_DATE,OCC_DATE,WEEKDAY,HATE_CRIME,HC_MOTIVATION,HC_STATUS,OFF_CATEG,NB_NAME_EN,SECTOR,DIVISION,CENSUS_TRC,WARD,COUNCILLOR,OFF_TYPE,geometry
0,1,1,2022,"Wed, 04 May 2022 00:00:00 GMT","Wed, 04 May 2022 00:00:00 GMT",Wednesday,Religion,Jewish,Hatecrime Happened,Public Incitement Of Hatred,Corkery,Sector 11,West,5050300.02,Ward 5 - West Carleton-March,Clarke Kelly,Criminal,None


In [488]:
total_hate = df_hate.groupby(['NB_NAME_EN', 'HATE_CRIME']).agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
total_hate.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Count'}, inplace=True)

agg_hate = total_hate.pivot(
    index='Neighbourhood',
    columns='HATE_CRIME',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

print(f'We had {total_hate["Count"].sum()} hate crimes across {agg_hate.shape[0]} neighbourhoods.')

agg_hate.head(1)

We had 1689 hate crimes across 105 neighbourhoods.


,Neighbourhood,Combination (More than 2 Motivations),Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Not Applicable,Other Similar Factor,Race/Ethnicity,Religion,Sexual Orientation,Unknown
0,Barrhaven,0,0,0,0,0,0,0,5,6,2,0


In [489]:
# collapse ambiguous/indeterminate categories into 'Other Hate Crimes'
ambiguous = [
    'Combination (More than 2 Motivations)',
    'Not Applicable',
    'Other Similar Factor',
    'Unknown',
]
agg_hate['Other Hate Crimes'] = agg_hate[ambiguous].sum(axis=1)
agg_hate = agg_hate.drop(columns=ambiguous)
agg_hate.head(1)

,Neighbourhood,Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Race/Ethnicity,Religion,Sexual Orientation,Other Hate Crimes
0,Barrhaven,0,0,0,0,5,6,2,0


#### Homicide
**Homicide → `agg_homicide`**: count per neighbourhood.

In [490]:
df_homicide.head(1)

,OBJECTID,YEAR,REP_DATE,OCC_DATE,WEEKDAY,OFF_CATEG,SECTOR,DIVISION,NB_NAME_EN,WARD,COUNCILLOR,geometry
0,1,2018,"Tue, 09 Jan 2018 05:00:00 GMT","Tue, 09 Jan 2018 05:00:00 GMT",Wednesday,Murder 1st Dgree,35,East,Hunt Club East - Western Community,Ward 16 - River,Riley Brockington,POINT Z (-75.67485 45.34915 0)


In [491]:
agg_homicide = df_homicide.groupby(['NB_NAME_EN']).agg({'OBJECTID': 'count'}) \
    .reset_index()

# rename the columns
agg_homicide.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Homicide'}, inplace=True)

print(f'We had {agg_homicide["Homicide"].sum()} homocides across {agg_homicide.shape[0]} neighbourhoods.')
agg_homicide.sort_values('Homicide', ascending=False).head()


We had 131 homocides across 46 neighbourhoods.


,Neighbourhood,Homicide
28,Lowertown,10
41,Vanier North,8
31,New Barrhaven - New Development - Stonebridge,8
15,Elmvale - Eastway - Riverview - Riverview Park...,7
42,West Centertown,6


#### Shootings
 **Shootings → `agg_shootings`**: count per neighbourhood.

In [492]:
df_shooting.head(1)

,OBJECTID,ID,REP_DATE,REP_HOUR,REP_YEAR,OCC_DATE,OCC_HOUR,OCC_YEAR,TOD,WEEKDAY,DOW,NB_NAME_EN,SECTOR,DIVISION,WARD_EN,COUNCILLOR,CENSUS_TRC,geometry
0,1,1,"Wed, 03 Jan 2018 05:00:00 GMT",2200,2018,"Wed, 03 Jan 2018 05:00:00 GMT",2200,2018,Evening,Wednesday,3,Elmvale - Eastway - Riverview - Riverview Park...,33,East,Ward 18 - Alta Vista,Marty Carr,5050008.00,POINT (-75.62188 45.38821)


In [493]:
agg_shootings = df_shooting.groupby('NB_NAME_EN').agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Shootings'}) \
    .reset_index()

# rename the columns
agg_shootings.rename(columns={'NB_NAME_EN':'Neighbourhood','OBJECTID': 'Shootings'}, inplace=True)


print(f'We had {agg_shootings["Shootings"].sum()} shootings across {agg_shootings.shape[0]} neighbourhoods')
agg_shootings.sort_values('Shootings', ascending=False).head()

We had 505 shootings across 94 neighbourhoods


,Neighbourhood,Shootings
73,Overbrook - McArthur,28
54,Ledbury - Heron Gate - Ridgemont - Elmwood,27
12,Byward Market,27
20,Centretown,22
55,Lowertown,21


#### Population
**Population → `df_neighbourhoods`**: neighbourhood boundaries file filtered to `ONS_ID`, `Name`, `POPEST`, `geometry`.

In [494]:
# filter and keeo only necessary columns
df_neighbourhoods = df_neighbourhoods.loc[:,['ONS_ID','Name','POPEST','geometry']]

print(f'We had {df_neighbourhoods["POPEST"].sum()} population across {df_neighbourhoods.shape[0]} neighbourhoods')
df_neighbourhoods.sort_values('POPEST', ascending=False).head()

We had 867146 population across 111 neighbourhoods


,ONS_ID,Name,POPEST,geometry
79,951,Stittsville,26674,"MULTIPOLYGON (((-75.94411 45.28927, -75.94426 ..."
16,24,Centretown,24994,"MULTIPOLYGON (((-75.70993 45.42249, -75.70995 ..."
103,937,Old Barrhaven East,22286,"POLYGON ((-75.7191 45.27663, -75.71863 45.2758..."
62,13,Bridlewood - Emerald Meadows,21101,"POLYGON ((-75.83849 45.28295, -75.83848 45.282..."
35,940,Overbrook - McArthur,19599,"POLYGON ((-75.63261 45.43266, -75.63203 45.432..."


#### Dispatched Calls for Service
**Calls for Service → `agg_police`**: filtered to priority levels 1–3 (dispatched responses only; priorities 4–7 are non-deployed). Mapped to severity labels (`Police_Critical` = life-threatening, `Police_High` = serious harm possible, `Police_Medium` = risk with delay). Counted per neighbourhood × severity, pivoted wide, then joined with `df_neighbourhoods` to attach geometry and population.

We shall aggregate by call priority. Here are the equivalent provided by the OPS
- 1: life-threatening
- 2: serious harm possible
- 3: risk with delay
- 4: mobile response
- 5: broadcast only
- 6: alternate response
- 7: property queue

In [8]:
print(f'Number of calls priorities: {df_calls['PRIORITY'].nunique()}')

df_calls.head(1)

Number of calls priorities: 7


,OBJECTID,PRIORITY,INIT_BY,RCVD_YEAR,RCVD_DATE,NB_ID,TOD,RCVD_HOUR,DOW,geometry
0,1,4,C,2021,"Fri, 01 Jan 2021 05:00:00 GMT",908.0,N,0,5,POINT (-75.68958 45.43053)


We are just going to keep 3 [1,2,3] priority levels - as these are when a service(s) is depolyed out to the neighbourhood.

In [538]:
# drop any priority other than 1,2,3
df_calls = df_calls[df_calls['PRIORITY'].isin([1, 2, 3])]

# map out the priority levels of police calls
priority_severity = {
    1: 'Police_Critical', # life-threatening
    2: 'Police_High', # serious harm possible
    3: 'Police_Medium', # risk with delay
}
df_calls['Priority Severity'] = df_calls['PRIORITY'].map(priority_severity)

In [539]:
# aggregate data by neighbourhood and priority severity
total_police = df_calls.groupby(['NB_ID', 'Priority Severity']).agg({'OBJECTID': 'count'}) \
    .rename(columns={'OBJECTID': 'Count'}) \
    .reset_index()

# pivot the data
agg_police = total_police.pivot(
    index='NB_ID',
    columns='Priority Severity',
    values='Count'
).fillna(0).astype(int).reset_index().rename_axis(None, axis=1)

agg_police.head(1)

print(f'We had calls across {agg_police.shape[0]} neighbourhoods')
agg_police.sort_values('Police_Critical', ascending=False).head(1)

We had calls across 111 neighbourhoods


,NB_ID,Police_Critical,Police_High,Police_Medium
11,24.0,1317,18859,28926


In [540]:
# we can now join with the population data as it has the neighbourhod id
agg_police['NB_ID'] = agg_police['NB_ID'].astype(int)
agg_police = agg_police.merge(
    df_neighbourhoods[['ONS_ID', 'Name', 'POPEST', 'geometry']].rename(columns={'ONS_ID': 'NB_ID'}),
    on='NB_ID',
    how='left'
)

# rename column
agg_police.rename(columns={'Name': 'Neighbourhood','POPEST':'Population'}, inplace=True)

# rearrange
agg_police = agg_police[['Neighbourhood','Population','Police_Critical', 'Police_High', 'Police_Medium', 'geometry']]
agg_police.head(1)

,Neighbourhood,Population,Police_Critical,Police_High,Police_Medium,geometry
0,Beacon Hill South - Cardinal Heights,7195,75,1037,2141,"POLYGON ((-75.58543 45.44887, -75.58545 45.448..."


### Merge all datasets into a single dataframe

`agg_crime` (113 neighbourhoods) is used as the left base. All other aggregated crime datasets are left-joined on `Neighbourhood` and missing values filled with 0:

```
agg_crime ← agg_auto ← agg_bike ← agg_hate ← agg_homicide ← agg_shootings
```

Twelve columns with ambiguous or low-signal content are summed into a single `Other` feature and the originals are dropped, reducing the merged dataframe from 30 to 19 columns:
- *From criminal offences:* `Commodification Of Sexual Activity`, `Fraud`, `Other Criminal Code`, `Prostitution`
- *From hate crimes:* `Gender`, `Immigrants/Newcomers to Canada`, `Language`, `Mental or Physical Disability`, `Race/Ethnicity`, `Religion`, `Sexual Orientation`, `Other Hate Crimes`


the population and call for service data use different column naming convensions, so we will first aggrage the following, handling the naming discrepancies and then join to a single dataframe for clustering.

step 1

In [541]:
# agg_crime has the most neighbourhoods (114), use it as the left base
dfs_to_merge = [agg_auto, agg_bike, agg_hate, agg_homicide, agg_shootings]

df_merged = agg_crime.copy()
for df in dfs_to_merge:
    df_merged = df_merged.merge(df, on='Neighbourhood', how='left')

df_merged = df_merged.fillna(0)
print(f'Merged dataset: {df_merged.shape}')
df_merged.head()

Merged dataset: (113, 30)


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Commodification Of Sexual Activity,Fraud,Mischief,Offensive Weapons,Other Criminal Code,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Prostitution,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom,Auto Theft,Bike Theft,Gender,Immigrants/Newcomers to Canada,Language,Mental or Physical Disability,Race/Ethnicity,Religion,Sexual Orientation,Other Hate Crimes,Homicide,Shootings
0,Barrhaven,3,208,0,79,0,463,258,10,87,171,0,0,59,346,9,3,4,103.0,61.0,0.0,0.0,0.0,0.0,5.0,6.0,2.0,0.0,3.0,2.0
1,Bayshore,3,278,1,129,0,500,244,6,152,203,11,0,73,1722,42,0,5,119.0,90.0,1.0,1.0,0.0,0.0,6.0,1.0,2.0,2.0,0.0,8.0
2,Beacon Hill South - Cardinal Heights,5,266,1,107,0,409,210,9,137,187,4,0,76,598,23,0,9,76.0,60.0,0.0,0.0,0.0,0.0,9.0,1.0,3.0,0.0,0.0,3.0
3,Beaverbrook,1,87,1,34,0,118,97,0,53,85,0,0,27,113,10,0,1,23.0,33.0,1.0,0.0,0.0,0.0,4.0,7.0,0.0,0.0,0.0,0.0
4,Beechwood Cemetery,1,1,0,1,0,3,3,0,2,0,1,0,0,4,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


We can combine the following petty crime features into a 'Other' feature based on their nature.

In [542]:
other_cols = [
    'Commodification Of Sexual Activity', 'Fraud', 'Other Criminal Code', 'Prostitution',
    'Gender', 'Immigrants/Newcomers to Canada', 'Language', 'Mental or Physical Disability',
    'Race/Ethnicity', 'Religion', 'Sexual Orientation', 'Other Hate Crimes',
]
df_merged['Other'] = df_merged[other_cols].sum(axis=1)
df_merged = df_merged.drop(columns=other_cols)
print(f'df_merged shape after combining: {df_merged.shape}')
df_merged.head(2)

df_merged shape after combining: (113, 19)


,Neighbourhood,Arson,Assaults,Attempting The Commission Of A Capital Crime,Break and Enter,Mischief,Offensive Weapons,Other Violations Involving Violence Or The Threat Of Violence,Possession / Trafficking Stolen Goods,Sexual Violations,Theft $5000 and Under,Theft Over $5000,Violations Causing Death,Violations Resulting In The Deprivation Of Freedom,Auto Theft,Bike Theft,Homicide,Shootings,Other
0,Barrhaven,3,208,0,79,258,10,171,0,59,346,9,3,4,103.0,61.0,3.0,2.0,563.0
1,Bayshore,3,278,1,129,244,6,203,11,73,1722,42,0,5,119.0,90.0,0.0,8.0,665.0


The crime dataset and the calls-for-service dataset use different neighbourhood naming conventions (69 of 113 match exactly out of the box). Mismatches are resolved in three passes:

In [543]:
# preview mismatches
map_names = set(agg_police['Neighbourhood'])
df_names = set(df_merged['Neighbourhood'])

print(f"Matched: {len(map_names & df_names)}")
print(f"\nIn map but not in df:\n{sorted(map_names - df_names)}")
print(f"\nIn df but not in map:\n{sorted(df_names - map_names)}")

Matched: 69

In map but not in df:
['Bayshore - Belltown', 'Borden Farm - Fisher Glen', "Brookside - Briarbrook - Morgan's Grant", 'Cardinal Creek', 'Chapel Hill North', 'Chapel Hill South', 'Chapman Mills', 'Chatelaine Village', 'Cityview - Crestview - Meadowlands', 'Convent Glen - Orléans Woods', 'Edwards - Carlsbad Springs', 'Elmvale - Canterbury', 'Fallingbrook', 'Findlay Creek', 'Greenbelt', 'Iris - Queensway Terrance South', 'Island Park - Wellington Village', 'Kanata Lakes - Arcardia', 'Ledbury - Heron Gate - Ridgemont', 'Manor Park', 'Manotick', 'Marlborough', 'Munster - Ashton', 'Navan - Sarsfield', 'North Gower - Kars', 'Old Barrhaven East', 'Old Barrhaven West', 'Old Ottawa East', 'Old Ottawa South', 'Osgoode - Vernon', 'Parkwood Hills - Stewart Farm', 'Portobello South', 'Queenswood Heights', 'Rideau Crest - Davidson Heights', 'Riverview', 'Rockcliffe Park', 'Sandy Hill', 'Skyline - Fisher Heights', 'South Keys - Greenboro West', "Stonebridge - Half Moon Bay - Heart's Desir

1. **Polygon unions**: Several crime-dataset neighbourhood names correspond to multiple map polygons. Their geometries are unioned in `agg_police` before any name matching (e.g. `Barrhaven` ← Old Barrhaven West + Old Barrhaven East; `Orléans Avalon…` ← Fallingbrook + Portobello South + Cardinal Creek).

In [ ]:
# df neighbourhood -> constituent map neighbourhoods whose geometries should be unioned
combinations = {
    'Barrhaven':                                                 ['Old Barrhaven West', 'Old Barrhaven East'],
    'New Barrhaven - New Development - Stonebridge':             ["Stonebridge - Half Moon Bay - Heart's Desire"],
    'Rockcliffe - Manor Park':                                   ['Rockcliffe Park', 'Manor Park'],
    'Chapman Mills - Rideau Crest - Davidson Heights':           ['Chapman Mills', 'Rideau Crest - Davidson Heights'],
    'Cityview - Skyline - Fisher Heights':                       ['Cityview - Crestview - Meadowlands', 'Skyline - Fisher Heights'],
    'Borden Farm - Stewart Farm - Parkwood Hills - Fisher Glen': ['Parkwood Hills - Stewart Farm', 'Borden Farm - Fisher Glen'],
    'Sandy Hill - Ottawa East':                                  ['Sandy Hill', 'Old Ottawa East'],
    'Riverside South - Leitrim':                                 ['Riverside South - Leitrim', 'Findlay Creek'],
    "Kanata Lakes - Marchwood Lakeside - Morgan's Grant - Kanata North Business Park": ['Kanata Lakes - Arcardia', "Brookside - Briarbrook - Morgan's Grant"],
    'Elmvale - Eastway - Riverview - Riverview Park West':       ['Riverview', 'Elmvale - Canterbury'],
    'Orléans Avalon - Notting Gate - Fallingbrook - Gardenway South': ['Fallingbrook', 'Portobello South', 'Cardinal Creek'],
}

# build all combined rows first (parts stay in agg_police during iteration),
# then remove all consumed parts in one pass — allows multiple keys to share the same parts
all_parts_to_remove = set()
combined_rows = []

for df_name, parts in combinations.items():
    subset = agg_police[agg_police['Neighbourhood'].isin(parts)]
    missing = set(parts) - set(subset['Neighbourhood'])
    if missing:
        print(f"Warning: parts not found for '{df_name}': {missing}")
        continue
    combined_rows.append(gpd.GeoDataFrame([{
        'Neighbourhood':   df_name,
        'Population':      int(subset['Population'].sum()),
        'Police_Critical': int(subset['Police_Critical'].sum()),
        'Police_High':     int(subset['Police_High'].sum()),
        'Police_Medium':   int(subset['Police_Medium'].sum()),
        'geometry':        gpd.GeoSeries(subset['geometry']).union_all(),
    }], crs=df_neighbourhoods.crs))
    all_parts_to_remove.update(parts)

agg_police = agg_police[~agg_police['Neighbourhood'].isin(all_parts_to_remove)]
agg_police = pd.concat([agg_police] + combined_rows, ignore_index=True)

print(f"agg_police shape: {agg_police.shape}")

agg_police shape: (100, 6)


2. **Manual overrides**: A lookup table of confirmed low-confidence or ambiguous matches is applied first (e.g. `Bayshore` → `Bayshore - Belltown`, `CFB Rockcliffe-NRC` → `Wateridge Village`, `Ottawa East` → `Sandy Hill - Ottawa East`).
3. **Fuzzy matching**: Remaining names matched using `rapidfuzz` token-sort ratio at a threshold of 70. Result: 104 of 113 names successfully mapped.

In [ ]:
# set the threshold 
THRESHOLD = 70

map_name_list = agg_police['Neighbourhood'].tolist()
df_name_list = df_merged['Neighbourhood'].tolist()

# manual overrides for confirmed low-confidence or ambiguous matches
manual_overrides = {
    'Bayshore':                                                              'Bayshore - Belltown',
    'Osgoode':                                                               'Osgoode - Vernon',
    'Sarsfield':                                                             'Navan - Sarsfield',
    'Russell - Edwards':                                                     'Edwards - Carlsbad Springs',
    'Munster':                                                               'Munster - Ashton',
    'CFB Rockcliffe-NRC':                                                    'Wateridge Village',
    'Island Park':                                                           'Island Park - Wellington Village',
    'Iris':                                                                  'Iris - Queensway Terrance South',
    'Orléans North West':  'Convent Glen - Orléans Woods',                                                  
    'Orléans Industrial':'Orléans Industrial',
    'Orléans Village - Chateauneuf':        'Orléans Village - Chateauneuf',
    'Orléans Chatelaine Village' :'Chatelaine Village',
    
    'Pierces Corners':'Marlborough',
    
    # Manotick split in crimes but single zone in calls
    'Manotick East':                                                         'Manotick',
    'Manotick West':                                                         'Manotick',
    'North Gower':                                                           'North Gower - Kars',
    # Ottawa East covers Old Ottawa East, which was merged into Sandy Hill - Ottawa East
    'Ottawa East':                                                           'Sandy Hill - Ottawa East',
    # Crestview - Meadowlands is part of the combined Cityview - Skyline zone
    'Crestview - Meadowlands':                                               'Cityview - Skyline - Fisher Heights',
}

fuzzy_map = {}
unmatched = []

for name in df_name_list:
    if name in manual_overrides:
        fuzzy_map[name] = manual_overrides[name]
        continue
    result = process.extractOne(name, map_name_list, scorer=fuzz.token_sort_ratio)
    if result and result[1] >= THRESHOLD:
        fuzzy_map[name] = result[0]
    else:
        unmatched.append((name, result))

print(f"Mapped:          {len(fuzzy_map)} / {len(df_name_list)}")
print(f"Still unmatched: {len(unmatched)}")
if unmatched:
    print("\nUnmatched (review manually):")
    for name, result in unmatched:
        print(f"  '{name}' -> best: '{result[0]}' ({result[1]:.0f})")

Mapped:          104 / 113
Still unmatched: 9

Unmatched (review manually):
  'Cummings' -> best: 'Carlington' (44)
  'Galetta' -> best: 'Laurentian' (47)
  'Greenbelt - Mer Bleue' -> best: 'Greenbelt' (60)
  'Greenbelt - Rideau River East' -> best: 'Carleton Heights - Rideauview' (59)
  'Greenbelt - Rideau River West' -> best: 'South Keys - Greenboro West' (57)
  'Greenbelt - Shirleys Bay' -> best: 'Briar Green - Leslie Park' (57)
  'Greenbelt - Stony Swamp' -> best: 'Greenbelt' (56)
  'Greenbelt SouthEast' -> best: 'South Keys - Greenboro West' (65)
  'Woodroffe - Lincoln Heights' -> best: 'Rothwell Heights - Beacon Hill North' (54)


The crime dataset is inner-joined with `agg_police` on the resolved keys, attaching population, geometry, and police call counts to each row.

In [ ]:
# keep only the mapped neighbourhoods and attach their agg_police equivalent name
df_mapped = df_merged[df_merged['Neighbourhood'].isin(fuzzy_map)].copy()
df_mapped['police_key'] = df_mapped['Neighbourhood'].map(fuzzy_map)

# join with agg_police on the mapped key; Neighbourhood stays as the crime-dataset name
df_final = df_mapped.merge(
    agg_police.rename(columns={'Neighbourhood': 'police_key'}),
    on='police_key',
    how='inner'
).drop(columns='police_key')

print(f'Final dataset: {df_final.shape}')

Final dataset: (104, 24)


Nine Greenbelt sub-areas (e.g. `Greenbelt - Mer Bleue`, `Greenbelt SouthEast`) are excluded — their boundaries are ambiguous across datasets and they contain no residential population relevant to the safety analysis. The remaining 9 unmatched names are rural/edge-case areas with no counterpart in the calls-for-service data.

In [548]:
# drop Greenbelt neighbourhoods — boundaries are ambiguous/indecisive
df_final = df_final[~df_final['Neighbourhood'].str.contains('Greenbelt', case=False, na=False)]
print(f'df_final shape after dropping Greenbelt: {df_final.shape}')

df_final shape after dropping Greenbelt: (104, 24)


#### Export data for model building and cluster analysis

In [549]:
df_final.to_csv('CleanedData.csv', index=False)
print('Cleaned data saved.')

Cleaned data saved.
